# Etapa 5b — Combinación de filtros sobre el verde

**Pregunta.** En la etapa 5, combinar **mapas de color** (tercera componente y verde) no superó al verde con el detector de líneas. Falta la forma clásica de usar varios **mapas de características**: combinar los distintos filtros de realce sobre el mismo verde. ¿Esa combinación segmenta mejor que el verde con el detector de líneas solo?

**Entradas** (a la mitad de resolución, que en la verificación del 31/10 resultó mejor que 1/4 para el verde): el verde con el fondo corregido (gaussiana, σ = 0,02 del diámetro, como en la etapa 5) y las respuestas de los cinco filtros de la etapa 4 (`src/enhance.py`): sombrero de copa, detector de líneas, filtro adaptado, Gabor y Frangi. Cada mapa se estandariza dentro del campo detectado.

**Combinación.** Regresión logística por píxel (`src/stats.py`, `fit_logistic`), ajustada en cada vuelta con las personas de entrenamiento (5 grupos de personas), con 20.000 píxeles de vaso y 20.000 de fondo por imagen.

**Umbral** (tres reglas, para la combinación y para el detector de líneas solo):
- Otsu en cada imagen (la regla de la etapa 5);
- una fracción fija de píxeles del campo marcada como vaso, elegida en entrenamiento;
- un valor fijo, elegido en entrenamiento.

Las dos últimas se eligen maximizando el Matthews medio de las imágenes de entrenamiento, estimado en la muestra fija de evaluación con el fondo ponderado (`weighted_matthews`).

**Cómo se mide.** Con las personas que quedan afuera, a resolución original y con todos los píxeles del campo de visión de la base: Matthews, Dice de línea central y sensibilidad de línea central en vasos finos.

**Cómo se decide.** La referencia es el verde con el detector de líneas y Otsu (la candidata de la etapa 5, a la mitad de resolución). Se elige la más simple entre las no inferiores a la mejor en Matthews (margen −0,03, desvío 14.4). Orden de simplicidad: detector de líneas con Otsu, con fracción, con valor fijo; después la combinación con las mismas tres reglas. Si la combinación gana con un intervalo que no toca el cero, la pregunta principal se responde a favor.


In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images, weighted_matthews
from src.split import person_folds
from src.stats import fit_logistic, percentile_interval, person_bootstrap_means, predict_logistic

ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
FACTOR = 2
FILTERS = ["sombrero de copa", "detector de líneas", "filtro adaptado", "Gabor", "Frangi"]
FEATS = ["verde corregido"] + FILTERS
REF = FEATS.index("detector de líneas")
N_TRAIN, N_THR = 20_000, 30_000
BG_W = 1 / C.EVAL_BG_FRACTION

def features(rgb):
    """Los seis mapas estandarizados, a 1/FACTOR de la resolución, y el campo detectado."""
    import numpy as np
    from src.color import standardize
    from src.enhance import apply_filter, prepare_maps
    from src.preprocess import corregir_fondo
    maps, region, d = prepare_maps(rgb, ["Verde"], FACTOR)
    m = corregir_fondo(maps["Verde"], region, 0.02 * d)
    out = [m] + [apply_filter(f, m, d) for f in FILTERS]
    return np.stack([standardize(x, region) for x in out]), region

## Pasada 1: muestras de entrenamiento y de elección de umbral

In [2]:
def pass1(image_id):
    import numpy as np
    from src.data import load_image
    from src.evaluation import load_sample, sample_values
    s = load_sample(image_id)
    F, _ = features(load_image(image_id))
    V = np.stack([sample_values(f, s) for f in F], axis=1).astype(np.float32)
    rng = np.random.default_rng(C.SEED + sum(map(ord, image_id)))
    pos, neg = np.flatnonzero(s.truth), np.flatnonzero(~s.truth)
    train = np.r_[rng.choice(pos, N_TRAIN, replace=False), rng.choice(neg, N_TRAIN, replace=False)]
    thr = rng.choice(len(s.truth), N_THR, replace=False)       # proporciones de la muestra
    return {"image": image_id, "Xtr": V[train], "ytr": s.truth[train],
            "Xthr": V[thr], "ythr": s.truth[thr], "wthr": np.where(s.truth[thr], 1.0, BG_W)}

t = time.time()
P1 = {r["image"]: r for r in map_images(pass1, ids, workers=4)}
print(f"Pasada 1: {time.time() - t:.0f} s")

Pasada 1: 314 s


## Ajuste por vuelta: coeficientes y umbrales con las personas de entrenamiento

In [3]:
def wquantile(v, w, q):
    o = np.argsort(v); cw = np.cumsum(w[o]) / w.sum()
    return v[o][np.searchsorted(cw, q)]

FRACS = np.round(np.arange(0.05, 0.20, 0.01), 2)

def best_rules(score_by_img, train_imgs):
    """Fracción y valor fijo que maximizan el Matthews medio (ponderado) en entrenamiento."""
    allv = np.concatenate([score_by_img[i] for i in train_imgs])
    grid = np.quantile(allv, np.linspace(0.70, 0.995, 60))
    best_f = max(FRACS, key=lambda f: np.mean([
        weighted_matthews(score_by_img[i] > wquantile(score_by_img[i], P1[i]["wthr"], 1 - f), P1[i]["ythr"], P1[i]["wthr"])
        for i in train_imgs]))
    best_t = max(grid, key=lambda t: np.mean([
        weighted_matthews(score_by_img[i] > t, P1[i]["ythr"], P1[i]["wthr"]) for i in train_imgs]))
    return float(best_f), float(best_t)

fit = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    X = np.concatenate([P1[i]["Xtr"] for i in tr]); y = np.concatenate([P1[i]["ytr"] for i in tr])
    w = fit_logistic(X, y, ridge=1e-3)
    combo = {i: predict_logistic(P1[i]["Xthr"], w) for i in ids}
    ref = {i: P1[i]["Xthr"][:, REF].astype(np.float64) for i in ids}
    fit[k] = {"w": w, "combo": best_rules(combo, tr), "ref": best_rules(ref, tr)}
    print(f"vuelta {k}: fracción y valor fijo — combinación {fit[k]['combo']}, detector {fit[k]['ref']}")
coef = pd.DataFrame([fit[k]["w"][1:] for k in range(5)], columns=FEATS)
print("\nCoeficientes de la regresión (mapas estandarizados), por vuelta:")
coef.round(3)

vuelta 0: fracción y valor fijo — combinación (0.09, 0.8318498526598735), detector (0.09, 0.9709893512725831)


vuelta 1: fracción y valor fijo — combinación (0.09, 0.832132988689518), detector (0.09, 0.9760855031013493)


vuelta 2: fracción y valor fijo — combinación (0.09, 0.8319353664943507), detector (0.09, 0.9283578765392304)


vuelta 3: fracción y valor fijo — combinación (0.09, 0.8326865240132962), detector (0.09, 0.9507869529724122)


vuelta 4: fracción y valor fijo — combinación (0.09, 0.8229098076026254), detector (0.09, 0.945077468752861)

Coeficientes de la regresión (mapas estandarizados), por vuelta:


,verde corregido,sombrero de copa,detector de líneas,filtro adaptado,Gabor,Frangi
0,0.857,0.481,2.238,0.032,0.433,-1.246
1,0.828,0.553,2.174,0.059,0.441,-1.212
2,0.760,0.508,2.205,0.064,0.444,-1.221
3,0.808,0.504,2.195,-0.034,0.459,-1.157
4,0.749,0.519,2.128,-0.029,0.487,-1.129


## Pasada 2: máscaras de las personas de afuera, a resolución original

In [4]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.segment import segmentar_fraccion, segmentar_otsu
    from src.stats import predict_logistic
    from src.vessels import skeleton_widths
    k = int(folds[image_id]); f = fit[k]
    rgb = load_image(image_id)
    ref_mask, fov = load_vessels(image_id), load_fov(image_id)
    F, _ = features(rgb)
    combo = predict_logistic(F.reshape(len(F), -1).T, f["w"]).reshape(F.shape[1:]).astype(np.float32)
    scores = {"combinación": upscale(combo, fov.shape), "detector de líneas": upscale(F[REF], fov.shape)}
    auto = detect_fov_robust(rgb)
    skel, wd = skeleton_widths(ref_mask & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    rows = []
    for name, sc in scores.items():
        key = "combo" if name == "combinación" else "ref"
        masks = {"Otsu": segmentar_otsu(sc, auto)[0],
                 "fracción": segmentar_fraccion(sc, auto, f[key][0])[0],
                 "valor fijo": (sc > f[key][1]) & auto}
        for rule, m in masks.items():
            rows.append({"image": image_id, "mapa": name, "regla": rule,
                         "matthews": matthews(m, ref_mask, fov), "dice_linea_central": centerline_dice(m, ref_mask, fov),
                         "finos": m[thin].mean()})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=4) for r in rows])
R.to_csv(ROOT / "results" / "05b_combinacion_filtros.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

Pasada 2: 429 s


## Resultados

In [5]:
R["candidata"] = R["mapa"] + " + " + R["regla"]
ORDER = [f"detector de líneas + {r}" for r in ("Otsu", "fracción", "valor fijo")] + \
        [f"combinación + {r}" for r in ("Otsu", "fracción", "valor fijo")]
W = R.pivot(index="image", columns="candidata", values="matthews")[ORDER]
best = W.mean().idxmax()
diffs = W.sub(W[best], axis=0)
lo, hi = percentile_interval(person_bootstrap_means(diffs, dev["person"]))
ref = "detector de líneas + Otsu"
dref = W.sub(W[ref], axis=0)
lo_r, hi_r = percentile_interval(person_bootstrap_means(dref, dev["person"]))
table = pd.DataFrame({
    "Matthews": W.mean(),
    "Dice línea central": R.pivot(index="image", columns="candidata", values="dice_linea_central")[ORDER].mean(),
    "finos": R.pivot(index="image", columns="candidata", values="finos")[ORDER].mean(),
    "vs referencia": dref.mean(), "ref. inferior": lo_r, "ref. superior": hi_r,
    "vs mejor": diffs.mean(), "no inferior": lo >= -C.NONINF_MARGIN_MATTHEWS})
choice = next(c for c in ORDER if table.loc[c, "no inferior"])
print(f"Mejor: {best}; elegida (la más simple no inferior): {choice}")
table.round(4)

Mejor: combinación + valor fijo; elegida (la más simple no inferior): detector de líneas + fracción


,Matthews,Dice línea central,finos,vs referencia,ref. inferior,ref. superior,vs mejor,no inferior
candidata,,,,,,,,
detector de líneas + Otsu,0.6887,0.6402,0.1199,0.0000,0.0000,0.0000,-0.0335,False
detector de líneas + fracción,0.7022,0.6883,0.2190,0.0135,0.0063,0.0210,-0.0200,True
detector de líneas + valor fijo,0.7040,0.6895,0.2194,0.0153,0.0074,0.0228,-0.0182,True
combinación + Otsu,0.6661,0.6614,0.5619,-0.0226,-0.0454,-0.0035,-0.0561,False
combinación + fracción,0.7194,0.7104,0.2631,0.0307,0.0219,0.0393,-0.0028,True
combinación + valor fijo,0.7222,0.7104,0.2411,0.0335,0.0252,0.0411,0.0000,True


In [6]:
print("Por grupo (Matthews medio):")
R.join(dev["group"], on="image").pivot_table(index="candidata", columns="group", values="matthews").loc[ORDER].round(3)

Por grupo (Matthews medio):


group,dr,g,h
candidata,,,
detector de líneas + Otsu,0.658,0.698,0.709
detector de líneas + fracción,0.660,0.705,0.742
detector de líneas + valor fijo,0.659,0.708,0.744
combinación + Otsu,0.596,0.663,0.739
combinación + fracción,0.671,0.723,0.764
combinación + valor fijo,0.674,0.730,0.763


## Decisión (escrita después de correr, 31/10)

**Resultados con las personas de afuera** (resolución original, todos los píxeles):

| Candidata | Matthews | Dice de línea central | Vasos finos |
|---|---|---|---|
| Detector de líneas + Otsu (referencia de la etapa 5) | 0,689 | 0,640 | 12 % |
| Detector de líneas + fracción 9 % | 0,702 | 0,688 | 22 % |
| **Combinación + fracción 9 %** | **0,719** | **0,710** | **26 %** |
| Combinación + valor fijo | 0,722 | 0,710 | 24 % |

- **Otsu era parte del problema:** con el mismo detector, marcar el 9 % de los píxeles más altos sube Matthews 0,014 y casi duplica los vasos finos encontrados.
- **Combinar los filtros mejora al filtro solo** con la misma regla de umbral: Matthews +0,017 (intervalo +0,014 a +0,020), vasos finos +0,044 (+0,040 a +0,049), Dice de línea central +0,022 (+0,018 a +0,026). Gana en las 30 imágenes y en las 18 personas (Wilcoxon por persona, p = 0,000008), en los tres grupos y en los dos niveles de compresión. La fracción elegida fue 9 % en las 5 vueltas.
- La regresión pesa sobre todo el detector de líneas, después el verde corregido, el sombrero de copa y Gabor; Frangi entra con peso negativo y el filtro adaptado casi no pesa.

**Se adopta la combinación con la fracción fija como algoritmo final (decisión de criterio, desvío 14.5 del plan).** La regla 5.3 (la más simple entre las no inferiores, margen −0,03) habría elegido el detector solo con la fracción, porque la ventaja (+0,017) es menor que el margen. Se aparta de la regla porque la ventaja es firme en todas las medidas y grupos, y porque es justamente la pregunta principal del trabajo: con el detector solo como algoritmo final, el algoritmo y la referencia coincidirían y la pregunta quedaría vacía. **Referencia clásica** (plan, sección 10): el detector de líneas sobre el verde con la misma regla de umbral.

**Sesgos que se declaran:**
1. Los valores absolutos son optimistas: la validación cruzada cubre los pesos y los umbrales, pero el mapa, la corrección del fondo, los filtros, sus escalas y la mitad de resolución se eligieron con estas mismas 30 imágenes. La diferencia entre la combinación y la referencia se ve menos afectada, porque las dos comparten esas decisiones. La estimación honesta será la del conjunto de prueba.
2. La fracción fija supone que los vasos ocupan alrededor del 9 % del campo, un valor de esta base y de cómo marcaron los expertos (11 % en sanos, 8 % en los otros grupos). Puede no valer con otra cámara o mucha patología. El umbral fijo sobre la probabilidad no lo supone y dio lo mismo (0,722): las dos reglas se comparan en la etapa 8 con imágenes que simulan otras cámaras, y se congela la que aguante mejor.
3. La ganancia es mayor en sanos (+0,022) que en retinopatía (+0,012); parte puede venir del ancho dibujado, pero el Dice de línea central, que no depende de él, también mejora.
4. La adopción se decidió después de ver los resultados.
